<a href="https://colab.research.google.com/github/rafiebazine/rag-image-captioning/blob/raouf/phase_01_data_exploration/Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget -q --show-progress http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!wget -q --show-progress http://images.cocodataset.org/zips/train2017.zip
!wget -q --show-progress http://images.cocodataset.org/zips/val2017.zip

annotations_trainva 100%[===================>] 241.19M  66.4MB/s    in 3.9s    
train2017.zip       100%[===================>]  18.01G  52.9MB/s    in 5m 41s  
val2017.zip         100%[===================>] 777.80M  74.5MB/s    in 11s     


In [2]:
print("Extracting annotations...")
!unzip -q annotations_trainval2017.zip

Extracting annotations...


In [3]:
print("Extracting training dataset (this may take a while)...")
!unzip -q train2017.zip

Extracting training dataset (this may take a while)...


In [4]:
print("Extracting validation dataset...")
!unzip -q val2017.zip

Extracting validation dataset...


In [10]:
import os
import json
from pathlib import Path

# Reference the paths from the kernel state
target_jsons = {
    'kb_10k': '/content/phase_01_data_exploration/    splitted_data_set/kb_10k.json',
    'val_100': '/content/phase_01_data_exploration/    splitted_data_set/val_100.json',
    'test_100': '/content/phase_01_data_exploration/    splitted_data_set/test_100.json',
    'test_1000': '/content/phase_01_data_exploration/    splitted_data_set/test_1000.json'
}

train_img_dir = Path('/content/train2017')
val_img_dir = Path('/content/val2017')

# Count actual physical files
def count_files(dir_path):
    if not dir_path.exists():
        return 0
    return len([f for f in dir_path.glob('*') if f.is_file()])

print("=== Physical Directory File Counts ===")
print(f"Training Folder (train2017): {count_files(train_img_dir)} files")
print(f"Validation Folder (val2017): {count_files(val_img_dir)} files")

# Function to check if images in JSON physically exist on disk
def verify_coco_dataset(json_path, dataset_name, expected_count):
    print(f"\n--- Verifying {dataset_name} (Expected: {expected_count} images) ---")
    if not os.path.exists(json_path):
        print(f"Error: JSON file not found at '{json_path}'")
        return

    with open(json_path, 'r') as f:
        coco_data = json.load(f)

    # Support both list-style JSONs and dictionary-style COCO JSONs
    if isinstance(coco_data, list):
        images = coco_data
    elif isinstance(coco_data, dict):
        images = coco_data.get('images', [])
    else:
        images = []

    actual_json_count = len(images)
    print(f"Images in JSON: {actual_json_count}")

    missing_files = []
    for img in images:
        file_name = img.get('file_name')
        if not file_name:
            img_id = img.get('id')
            file_name = f"{str(img_id).zfill(12)}.jpg"

        # Check both potential directories since images could be split between train and val folders
        path_train = train_img_dir / file_name
        path_val = val_img_dir / file_name

        if not path_train.exists() and not path_val.exists():
            missing_files.append(file_name)

    if len(missing_files) == 0:
        print(f"✓ Success: All {actual_json_count} images are physically present on disk!")
    else:
        print(f"❌ Missing {len(missing_files)} images from disk.")
        print(f"Sample missing files: {missing_files[:10]}")

# Verify each split
verify_coco_dataset(target_jsons['kb_10k'], 'Training/Knowledge-base (kb_10k)', 10000)
verify_coco_dataset(target_jsons['val_100'], 'Validation (val_100)', 100)
verify_coco_dataset(target_jsons['test_1000'], 'Test (test_1000)', 1000)
verify_coco_dataset(target_jsons['test_100'], 'Test (test_100)', 100)

=== Physical Directory File Counts ===
Training Folder (train2017): 118287 files
Validation Folder (val2017): 5000 files

--- Verifying Training/Knowledge-base (kb_10k) (Expected: 10000 images) ---
Images in JSON: 10000
✓ Success: All 10000 images are physically present on disk!

--- Verifying Validation (val_100) (Expected: 100 images) ---
Images in JSON: 100
✓ Success: All 100 images are physically present on disk!

--- Verifying Test (test_1000) (Expected: 1000 images) ---
Images in JSON: 1000
✓ Success: All 1000 images are physically present on disk!

--- Verifying Test (test_100) (Expected: 100 images) ---
Images in JSON: 100
✓ Success: All 100 images are physically present on disk!


In [16]:

import json
import os
from collections import Counter

target_jsons = {
    'kb_10k': '/content/phase_01_data_exploration/splitted_data_set/kb_10k.json',
    'val_100': '/content/phase_01_data_exploration/splitted_data_set/val_100.json',
    'test_100': '/content/phase_01_data_exploration/splitted_data_set/test_100.json',
    'test_1000': '/content/phase_01_data_exploration/splitted_data_set/val_100.json'
}

EXPECTED_CAPTIONS = 5

print("=" * 90)
print("REFERENCE CAPTIONS VALIDATION")
print("=" * 90)

all_passed = True
dataset_results = {}

for dataset_name, json_path in target_jsons.items():

    print(f"\n{'-' * 90}")
    print(f"Dataset: {dataset_name}")
    print(f"File: {json_path}")
    print("-" * 90)

    if not os.path.exists(json_path):
        print("❌ FILE NOT FOUND")
        all_passed = False
        continue

    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    if not isinstance(data, list):
        print("❌ Invalid JSON structure: expected a list of images.")
        all_passed = False
        continue

    image_ids = [item.get('image_id') for item in data]
    file_names = [item.get('file_name') for item in data]

    id_counts = Counter(image_ids)
    filename_counts = Counter(file_names)

    duplicate_ids = {
        k: v for k, v in id_counts.items()
        if k is not None and v > 1
    }

    duplicate_filenames = {
        k: v for k, v in filename_counts.items()
        if k is not None and v > 1
    }

    caption_counts = Counter()
    invalid_captions = []
    missing_captions = []
    duplicate_captions = []

    for item in data:
        image_id = item.get('image_id')
        captions = item.get('captions')

        if not isinstance(captions, list):
            missing_captions.append(image_id)
            continue

        caption_counts[len(captions)] += 1

        if len(captions) != EXPECTED_CAPTIONS:
            invalid_captions.append({
                'image_id': image_id,
                'file_name': item.get('file_name'),
                'caption_count': len(captions)
            })

        # Check empty or non-string captions
        for i, caption in enumerate(captions):
            if not isinstance(caption, str) or not caption.strip():
                missing_captions.append({
                    'image_id': image_id,
                    'caption_index': i
                })

        # Check duplicate captions within the same image
        normalized = [
            c.strip().lower()
            for c in captions
            if isinstance(c, str)
        ]

        if len(normalized) != len(set(normalized)):
            duplicate_captions.append(image_id)

    valid_images = sum(
        1 for item in data
        if isinstance(item.get('captions'), list)
        and len(item['captions']) == EXPECTED_CAPTIONS
        and all(
            isinstance(c, str) and c.strip()
            for c in item['captions']
        )
    )

    total_images = len(data)
    total_captions = sum(
        len(item['captions'])
        for item in data
        if isinstance(item.get('captions'), list)
    )

    passed = (
        len(invalid_captions) == 0
        and len(missing_captions) == 0
        and len(duplicate_ids) == 0
        and len(duplicate_filenames) == 0
    )

    dataset_results[dataset_name] = {
        'total_images': total_images,
        'total_captions': total_captions,
        'valid_images': valid_images,
        'invalid_caption_counts': len(invalid_captions),
        'missing_captions': len(missing_captions),
        'duplicate_ids': len(duplicate_ids),
        'duplicate_filenames': len(duplicate_filenames),
        'duplicate_captions': len(duplicate_captions),
        'passed': passed
    }

    print(f"Total images:                 {total_images}")
    print(f"Total reference captions:     {total_captions}")
    print(f"Expected captions per image:  {EXPECTED_CAPTIONS}")
    print(f"Images with exactly 5 valid captions: {valid_images}")
    print(f"Caption count distribution:   {dict(sorted(caption_counts.items()))}")

    print("\nValidation checks:")
    print(f"{'✅' if not invalid_captions else '❌'} Incorrect caption counts: {len(invalid_captions)}")
    print(f"{'✅' if not missing_captions else '❌'} Missing/empty captions: {len(missing_captions)}")
    print(f"{'✅' if not duplicate_ids else '❌'} Duplicate image IDs: {len(duplicate_ids)}")
    print(f"{'✅' if not duplicate_filenames else '❌'} Duplicate filenames: {len(duplicate_filenames)}")
    print(f"{'⚠️' if duplicate_captions else '✅'} Images with duplicate captions: {len(duplicate_captions)}")

    if invalid_captions:
        print("\nImages with incorrect caption counts:")
        for item in invalid_captions[:20]:
            print(f"  Image ID: {item['image_id']} | "
                  f"File: {item['file_name']} | "
                  f"Captions: {item['caption_count']}")

    if missing_captions:
        print("\nImages with missing/empty captions:")
        for item in missing_captions[:20]:
            print(f"  {item}")

    if duplicate_ids:
        print("\nDuplicate image IDs:")
        print(dict(list(duplicate_ids.items())[:20]))

    if duplicate_filenames:
        print("\nDuplicate filenames:")
        print(dict(list(duplicate_filenames.items())[:20]))

    if duplicate_captions:
        print("\nImages with duplicate captions:")
        print(duplicate_captions[:20])

    print(f"\nDataset result: {'✅ PASSED' if passed else '❌ FAILED'}")

    if not passed:
        all_passed = False


# Final summary
print("\n" + "=" * 90)
print("FINAL SUMMARY")
print("=" * 90)

for name, result in dataset_results.items():
    print(
        f"{name:12} | "
        f"Images: {result['total_images']:6} | "
        f"Captions: {result['total_captions']:7} | "
        f"Valid: {result['valid_images']:6} | "
        f"{'✅ PASSED' if result['passed'] else '❌ FAILED'}"
    )

print("\n" + "=" * 90)
print(
    "OVERALL RESULT:",
    "✅ ALL DATASETS PASSED" if all_passed else "❌ VALIDATION ISSUES FOUND"
)
print("=" * 90)


REFERENCE CAPTIONS VALIDATION

------------------------------------------------------------------------------------------
Dataset: kb_10k
File: /content/phase_01_data_exploration/splitted_data_set/kb_10k.json
------------------------------------------------------------------------------------------
Total images:                 10000
Total reference captions:     50000
Expected captions per image:  5
Images with exactly 5 valid captions: 10000
Caption count distribution:   {5: 10000}

Validation checks:
✅ Incorrect caption counts: 0
✅ Missing/empty captions: 0
✅ Duplicate image IDs: 0
✅ Duplicate filenames: 0
⚠️ Images with duplicate captions: 10

Images with duplicate captions:
[126876, 374987, 278582, 409842, 397375, 514913, 441453, 26654, 157726, 404618]

Dataset result: ✅ PASSED

------------------------------------------------------------------------------------------
Dataset: val_100
File: /content/phase_01_data_exploration/splitted_data_set/val_100.json
-------------------------

In [21]:

import json
import os
from PIL import Image, UnidentifiedImageError
from collections import Counter

# ============================================================
# 1. CONFIGURATION
# ============================================================

target_jsons = {
    'kb_10k': '/content/phase_01_data_exploration/splitted_data_set/kb_10k.json',
    'val_100': '/content/phase_01_data_exploration/splitted_data_set/val_100.json',
    'test_100': '/content/phase_01_data_exploration/splitted_data_set/test_100.json',
    'test_1000': '/content/phase_01_data_exploration/splitted_data_set/test_1000.json'
}

# COCO image directories
IMAGE_DIRS = [
    '/content/train2017',
    '/content/val2017'
]

EXPECTED_CAPTIONS = 5

dataset_results = {}


# ============================================================
# 2. IMAGE LOOKUP FUNCTION
# ============================================================

def find_image(file_name):

    if not file_name:
        return None

    for directory in IMAGE_DIRS:

        image_path = os.path.join(directory, file_name)

        if os.path.isfile(image_path):
            return image_path

    return None


# ============================================================
# 3. CHECK IMAGE DIRECTORIES
# ============================================================

print("=" * 90)
print("IMAGE DIRECTORIES CHECK")
print("=" * 90)

for directory in IMAGE_DIRS:

    exists = os.path.isdir(directory)

    print(f"\nDirectory: {directory}")
    print(f"Exists: {'YES' if exists else 'NO'}")

    if exists:
        print(f"Number of files: {len(os.listdir(directory))}")
    else:
        print("WARNING: Directory not found!")


# ============================================================
# 4. DATASET VALIDATION
# ============================================================

for dataset_name, json_path in target_jsons.items():

    print("\n" + "=" * 90)
    print(f"DATASET: {dataset_name}")
    print("=" * 90)

    if not os.path.exists(json_path):
        print(f"❌ JSON FILE NOT FOUND: {json_path}")
        continue

    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    if not isinstance(data, list):
        print("❌ Invalid JSON structure: expected a list.")
        continue

    dataset_issues = []

    image_ids = [item.get('image_id') for item in data]

    filename_counts = Counter(
        item.get('file_name')
        for item in data
        if item.get('file_name')
    )

    id_counts = Counter(image_ids)

    duplicate_ids = {
        k for k, v in id_counts.items()
        if k is not None and v > 1
    }

    # --------------------------------------------------------
    # VALIDATE EACH IMAGE
    # --------------------------------------------------------

    for item in data:

        image_id = item.get('image_id')
        file_name = item.get('file_name')
        captions = item.get('captions')

        issues = []

        # 1. IMAGE EXISTENCE

        image_path = find_image(file_name)

        if image_path is None:
            issues.append('IMAGE_FILE_NOT_FOUND')

        # 2. IMAGE READABILITY AND DIMENSIONS

        actual_width = None
        actual_height = None

        if image_path is not None:

            try:

                with Image.open(image_path) as img:
                    img.verify()

                with Image.open(image_path) as img:

                    actual_width, actual_height = img.size

                    expected_width = item.get('width')
                    expected_height = item.get('height')

                    if (
                        expected_width is not None
                        and actual_width != expected_width
                    ):
                        issues.append('WIDTH_MISMATCH')

                    if (
                        expected_height is not None
                        and actual_height != expected_height
                    ):
                        issues.append('HEIGHT_MISMATCH')

            except (
                UnidentifiedImageError,
                OSError,
                ValueError
            ):
                issues.append('CORRUPTED_OR_UNREADABLE_IMAGE')

            except Exception as e:
                issues.append(f'IMAGE_VALIDATION_ERROR: {str(e)}')

        # 3. CAPTION VALIDATION

        if not isinstance(captions, list):

            issues.append('CAPTIONS_NOT_A_LIST')

        else:

            if len(captions) != EXPECTED_CAPTIONS:
                issues.append(
                    f'INCORRECT_CAPTION_COUNT_{len(captions)}'
                )

            for idx, caption in enumerate(captions):

                if not isinstance(caption, str):
                    issues.append(f'NON_STRING_CAPTION_{idx}')

                elif not caption.strip():
                    issues.append(f'EMPTY_CAPTION_{idx}')

            normalized_captions = [
                c.strip().lower()
                for c in captions
                if isinstance(c, str) and c.strip()
            ]

            if len(normalized_captions) != len(set(normalized_captions)):
                issues.append('DUPLICATE_CAPTIONS')

        # 4. METADATA VALIDATION

        if image_id is None:
            issues.append('MISSING_IMAGE_ID')

        if not file_name:
            issues.append('MISSING_FILENAME')

        if image_id in duplicate_ids:
            issues.append('DUPLICATE_IMAGE_ID')

        if file_name and filename_counts[file_name] > 1:
            issues.append('DUPLICATE_FILENAME')

        # 5. STORE RESULT IN MEMORY ONLY

        dataset_issues.append({
            'image_id': image_id,
            'file_name': file_name,
            'image_path': image_path,
            'actual_width': actual_width,
            'actual_height': actual_height,
            'status': 'INVALID' if issues else 'VALID',
            'issues': issues
        })

    # ========================================================
    # 5. DATASET SUMMARY
    # ========================================================

    total = len(dataset_issues)

    valid = sum(
        r['status'] == 'VALID'
        for r in dataset_issues
    )

    invalid = total - valid

    dataset_results[dataset_name] = {
        'total': total,
        'valid': valid,
        'invalid': invalid
    }

    print(f"\nTotal images:   {total}")
    print(f"Valid images:   {valid}")
    print(f"Invalid images: {invalid}")

    # Issue distribution

    issue_counts = Counter()

    for result in dataset_issues:

        for issue in result['issues']:
            issue_counts[issue] += 1

    print("\nIssue distribution:")

    if issue_counts:

        for issue, count in issue_counts.most_common():
            print(f"  {issue}: {count}")

    else:
        print("  No issues found.")

    # Display invalid samples

    invalid_samples = [
        r for r in dataset_issues
        if r['status'] == 'INVALID'
    ]

    if invalid_samples:

        print("\nFirst 10 invalid samples:")

        for result in invalid_samples[:10]:

            print(
                f"  ID: {result['image_id']} | "
                f"File: {result['file_name']} | "
                f"Issues: {', '.join(result['issues'])}"
            )


# ============================================================
# 6. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 90)
print("FINAL IMAGE-CAPTION INTEGRITY SUMMARY")
print("=" * 90)

for name, result in dataset_results.items():

    print(
        f"{name:12} | "
        f"Total: {result['total']:6} | "
        f"Valid: {result['valid']:6} | "
        f"Invalid: {result['invalid']:6}"
    )

total_images = sum(
    r['total'] for r in dataset_results.values()
)

total_valid = sum(
    r['valid'] for r in dataset_results.values()
)

total_invalid = sum(
    r['invalid'] for r in dataset_results.values()
)

print("-" * 90)
print(f"TOTAL IMAGES:   {total_images}")
print(f"VALID IMAGES:   {total_valid}")
print(f"INVALID IMAGES: {total_invalid}")

print("\n" + "=" * 90)
print("VALIDATION COMPLETED")
print("No reports or documentation files were generated.")
print("No original JSON files or images were modified.")
print("No samples were automatically removed.")
print("=" * 90)


IMAGE DIRECTORIES CHECK

Directory: /content/train2017
Exists: YES
Number of files: 118287

Directory: /content/val2017
Exists: YES
Number of files: 5000

DATASET: kb_10k

Total images:   10000
Valid images:   9990
Invalid images: 10

Issue distribution:
  DUPLICATE_CAPTIONS: 10

First 10 invalid samples:
  ID: 126876 | File: 000000126876.jpg | Issues: DUPLICATE_CAPTIONS
  ID: 374987 | File: 000000374987.jpg | Issues: DUPLICATE_CAPTIONS
  ID: 278582 | File: 000000278582.jpg | Issues: DUPLICATE_CAPTIONS
  ID: 409842 | File: 000000409842.jpg | Issues: DUPLICATE_CAPTIONS
  ID: 397375 | File: 000000397375.jpg | Issues: DUPLICATE_CAPTIONS
  ID: 514913 | File: 000000514913.jpg | Issues: DUPLICATE_CAPTIONS
  ID: 441453 | File: 000000441453.jpg | Issues: DUPLICATE_CAPTIONS
  ID: 26654 | File: 000000026654.jpg | Issues: DUPLICATE_CAPTIONS
  ID: 157726 | File: 000000157726.jpg | Issues: DUPLICATE_CAPTIONS
  ID: 404618 | File: 000000404618.jpg | Issues: DUPLICATE_CAPTIONS

DATASET: val_100

Total

In [23]:

import os
import json
import re
import hashlib
from datetime import datetime, timezone
from collections import defaultdict

# ============================================================
# 1. CONFIGURATION
# ============================================================

target_jsons = {
    "kb_10k": "/content/phase_01_data_exploration/splitted_data_set/kb_10k.json",
    "val_100": "/content/phase_01_data_exploration/splitted_data_set/val_100.json",
    "test_1000": "/content/phase_01_data_exploration/splitted_data_set/test_1000.json",
}

EXPECTED_SIZES = {
    "kb_10k": 10000,
    "val_100": 100,
    "test_1000": 1000,
}

# Use v2 because the previous v1 audit already exists.
SPLIT_VERSION = "v2"

OUTPUT_DIR = (
    "/content/phase_01_data_exploration/frozen_splits"
)

# False: identical caption text on different images is a warning.
# True: any normalized caption shared between splits becomes an error.
STRICT_CAPTION_TEXT_LEAKAGE = False

# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def normalize_id(value):
    if value is None:
        return None
    return str(value).strip()


def normalize_filename(value):
    if not isinstance(value, str):
        return None
    return os.path.basename(value.strip()).lower()


def normalize_caption(caption):
    if not isinstance(caption, str):
        return None

    caption = caption.lower().strip()
    caption = re.sub(r"\s+", " ", caption)
    caption = re.sub(r"[^\w\s]", "", caption)

    return caption.strip()


def load_json_dataset(path):
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Dataset not found: {path}")

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(
            f"Expected a JSON list in {path}, "
            f"but received {type(data).__name__}"
        )

    return data


def sha256_ids(ids):
    content = json.dumps(
        ids,
        ensure_ascii=False,
        separators=(",", ":")
    )

    return hashlib.sha256(
        content.encode("utf-8")
    ).hexdigest()


def find_caption_owners(data):
    """
    Maps each normalized caption to the image IDs
    whose records contain that caption.
    """
    owners = defaultdict(set)

    for record in data:
        if not isinstance(record, dict):
            continue

        image_id = normalize_id(record.get("image_id"))

        if image_id is None:
            continue

        captions = record.get("captions", [])

        if not isinstance(captions, list):
            continue

        for caption in captions:
            normalized = normalize_caption(caption)

            if normalized:
                owners[normalized].add(image_id)

    return owners


def find_caption_overlap(owners_a, owners_b):
    """
    Finds normalized caption strings occurring in both splits.
    """
    common_captions = set(owners_a.keys()) & set(owners_b.keys())

    examples = []

    for caption in sorted(common_captions):
        examples.append({
            "caption": caption,
            "split_a_image_ids": sorted(owners_a[caption]),
            "split_b_image_ids": sorted(owners_b[caption]),
        })

    return {
        "count": len(common_captions),
        "examples": examples[:10],
    }


def filter_retrieval_candidates(
    query_image_id,
    candidate_image_ids
):
    """
    Prevents an image from being retrieved as its own match.
    Use this function in your actual retrieval pipeline.
    """
    query_image_id = normalize_id(query_image_id)

    return [
        candidate_id
        for candidate_id in candidate_image_ids
        if normalize_id(candidate_id) != query_image_id
    ]


def write_json_exclusive(path, content):
    """
    Creates a new file without overwriting an existing file.
    """
    with open(path, "x", encoding="utf-8") as f:
        json.dump(
            content,
            f,
            indent=4,
            ensure_ascii=False
        )


# ============================================================
# 3. LOAD DATASETS
# ============================================================

print("=" * 65)
print("LOADING DATASETS")
print("=" * 65)

datasets = {}

for split_name, path in target_jsons.items():
    datasets[split_name] = load_json_dataset(path)

    print(
        f"{split_name}: "
        f"{len(datasets[split_name]):,} images loaded"
    )


# ============================================================
# 4. INITIALIZE AUDIT
# ============================================================

errors = []
warnings = []

audit = {
    "version": SPLIT_VERSION,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "datasets": {},
    "checks": {},
    "errors": errors,
    "warnings": warnings,
}


# ============================================================
# 5. CHECK EXPECTED DATASET SIZES
# ============================================================

print("\n" + "=" * 65)
print("CHECKING DATASET SIZES")
print("=" * 65)

for split_name, expected_size in EXPECTED_SIZES.items():

    actual_size = len(datasets[split_name])

    passed = actual_size == expected_size

    audit["checks"][f"{split_name}_expected_size"] = {
        "expected": expected_size,
        "actual": actual_size,
        "passed": passed,
    }

    if not passed:
        errors.append(
            f"{split_name}: expected {expected_size} images, "
            f"found {actual_size}."
        )

    print(
        f"{split_name}: {actual_size:,} / "
        f"{expected_size:,} "
        f"{'PASS' if passed else 'FAIL'}"
    )


# ============================================================
# 6. CHECK RECORDS, IMAGE IDS, FILENAMES AND CAPTIONS
# ============================================================

print("\n" + "=" * 65)
print("CHECKING RECORDS")
print("=" * 65)

split_ids = {}
split_filenames = {}

for split_name, data in datasets.items():

    image_ids = []
    filenames = []

    invalid_records = 0
    missing_ids = 0
    missing_filenames = 0
    invalid_captions = 0
    wrong_caption_counts = 0
    duplicate_captions = 0

    for index, record in enumerate(data):

        if not isinstance(record, dict):
            invalid_records += 1
            continue

        image_id = normalize_id(record.get("image_id"))
        filename = normalize_filename(record.get("file_name"))

        if image_id is None:
            missing_ids += 1
        else:
            image_ids.append(image_id)

        if filename is None:
            missing_filenames += 1
        else:
            filenames.append(filename)

        captions = record.get("captions")

        if not isinstance(captions, list):
            invalid_captions += 1
            continue

        if len(captions) != 5:
            wrong_caption_counts += 1

        valid_normalized_captions = []

        for caption in captions:
            normalized = normalize_caption(caption)

            if not isinstance(caption, str) or not caption.strip():
                invalid_captions += 1
            else:
                valid_normalized_captions.append(normalized)

        if len(valid_normalized_captions) != len(
            set(valid_normalized_captions)
        ):
            duplicate_captions += 1

    duplicate_ids = len(image_ids) - len(set(image_ids))
    duplicate_filenames = len(filenames) - len(set(filenames))

    split_ids[split_name] = image_ids
    split_filenames[split_name] = filenames

    checks = {
        "invalid_records": invalid_records,
        "missing_image_ids": missing_ids,
        "missing_filenames": missing_filenames,
        "duplicate_image_ids": duplicate_ids,
        "duplicate_filenames": duplicate_filenames,
        "invalid_captions": invalid_captions,
        "records_not_having_5_captions": wrong_caption_counts,
        "records_with_duplicate_captions": duplicate_captions,
    }

    audit["checks"][f"{split_name}_record_checks"] = checks

    print(f"\n{split_name}:")

    for check_name, count in checks.items():
        print(f"  {check_name}: {count}")

    if invalid_records:
        errors.append(
            f"{split_name}: contains {invalid_records} invalid records."
        )

    if missing_ids:
        errors.append(
            f"{split_name}: contains {missing_ids} missing image IDs."
        )

    if missing_filenames:
        errors.append(
            f"{split_name}: contains {missing_filenames} missing filenames."
        )

    if duplicate_ids:
        errors.append(
            f"{split_name}: contains {duplicate_ids} duplicate image IDs."
        )

    if duplicate_filenames:
        errors.append(
            f"{split_name}: contains {duplicate_filenames} duplicate filenames."
        )

    if invalid_captions:
        errors.append(
            f"{split_name}: contains {invalid_captions} invalid captions."
        )

    if wrong_caption_counts:
        errors.append(
            f"{split_name}: {wrong_caption_counts} records "
            "do not contain exactly 5 captions."
        )

    if duplicate_captions:
        warnings.append(
            f"{split_name}: {duplicate_captions} records contain "
            "duplicate captions."
        )


# ============================================================
# 7. CHECK IMAGE ID OVERLAP BETWEEN SPLITS
# ============================================================

print("\n" + "=" * 65)
print("CHECKING IMAGE ID OVERLAP")
print("=" * 65)

split_pairs = [
    ("kb_10k", "val_100"),
    ("kb_10k", "test_1000"),
    ("val_100", "test_1000"),
]

for split_a, split_b in split_pairs:

    overlap = (
        set(split_ids[split_a])
        & set(split_ids[split_b])
    )

    passed = len(overlap) == 0

    audit["checks"][
        f"{split_a}_vs_{split_b}_image_id_overlap"
    ] = {
        "overlap_count": len(overlap),
        "examples": sorted(overlap)[:10],
        "passed": passed,
    }

    print(
        f"{split_a} vs {split_b}: "
        f"{len(overlap)} overlapping image IDs "
        f"{'PASS' if passed else 'FAIL'}"
    )

    if not passed:
        errors.append(
            f"Image ID overlap between {split_a} and {split_b}: "
            f"{len(overlap)} shared IDs."
        )


# ============================================================
# 8. CHECK FILENAME OVERLAP BETWEEN SPLITS
# ============================================================

print("\n" + "=" * 65)
print("CHECKING FILENAME OVERLAP")
print("=" * 65)

for split_a, split_b in split_pairs:

    overlap = (
        set(split_filenames[split_a])
        & set(split_filenames[split_b])
    )

    passed = len(overlap) == 0

    audit["checks"][
        f"{split_a}_vs_{split_b}_filename_overlap"
    ] = {
        "overlap_count": len(overlap),
        "examples": sorted(overlap)[:10],
        "passed": passed,
    }

    print(
        f"{split_a} vs {split_b}: "
        f"{len(overlap)} overlapping filenames "
        f"{'PASS' if passed else 'FAIL'}"
    )

    if not passed:
        errors.append(
            f"Filename overlap between {split_a} and {split_b}: "
            f"{len(overlap)} shared filenames."
        )


# ============================================================
# 9. CHECK CAPTION TEXT OVERLAP
# ============================================================

print("\n" + "=" * 65)
print("CHECKING CAPTION TEXT OVERLAP")
print("=" * 65)

caption_owners = {
    split_name: find_caption_owners(data)
    for split_name, data in datasets.items()
}

caption_overlap_results = {}

for split_a, split_b in split_pairs:

    result = find_caption_overlap(
        caption_owners[split_a],
        caption_owners[split_b]
    )

    key = f"{split_a}_vs_{split_b}_caption_overlap"

    caption_overlap_results[key] = result

    print(
        f"\n{split_a} vs {split_b}: "
        f"{result['count']} shared normalized caption strings"
    )

    for example in result["examples"][:5]:
        print(f"  Caption: {example['caption']}")
        print(
            f"    {split_a} image IDs: "
            f"{example['split_a_image_ids']}"
        )
        print(
            f"    {split_b} image IDs: "
            f"{example['split_b_image_ids']}"
        )

    if result["count"] > 0:

        message = (
            f"{result['count']} normalized caption strings occur in "
            f"both {split_a} and {split_b}. This may be natural "
            "caption repetition across different images."
        )

        if STRICT_CAPTION_TEXT_LEAKAGE:
            errors.append(message)
        else:
            warnings.append(message)

audit["checks"]["caption_text_overlap"] = caption_overlap_results

audit["checks"]["strict_caption_text_leakage"] = {
    "enabled": STRICT_CAPTION_TEXT_LEAKAGE,
    "description": (
        "When enabled, any normalized caption string shared between "
        "splits causes the audit to fail."
    ),
}


# ============================================================
# 10. CHECK SELF-RETRIEVAL FILTER
# ============================================================

print("\n" + "=" * 65)
print("CHECKING SELF-RETRIEVAL FILTER")
print("=" * 65)

kb_candidate_ids = split_ids["kb_10k"]

self_retrieval_errors = []

for query_image_id in (
    split_ids["val_100"] + split_ids["test_1000"]
):

    filtered_candidates = filter_retrieval_candidates(
        query_image_id,
        kb_candidate_ids
    )

    if query_image_id in filtered_candidates:
        self_retrieval_errors.append(query_image_id)

self_retrieval_passed = len(self_retrieval_errors) == 0

audit["checks"]["self_retrieval_filter"] = {
    "passed": self_retrieval_passed,
    "queries_checked": (
        len(split_ids["val_100"])
        + len(split_ids["test_1000"])
    ),
    "self_retrieval_errors": self_retrieval_errors[:10],
}

print(
    "Self-retrieval filter:",
    "PASS" if self_retrieval_passed else "FAIL"
)

if not self_retrieval_passed:
    errors.append(
        "Self-retrieval filter failed for one or more query images."
    )


# ============================================================
# 11. ADD DATASET HASHES AND SPLIT INFORMATION
# ============================================================

for split_name, data in datasets.items():

    ordered_ids = split_ids[split_name]

    audit["datasets"][split_name] = {
        "source_json": target_jsons[split_name],
        "image_count": len(data),
        "image_id_sha256": sha256_ids(ordered_ids),
    }


# ============================================================
# 12. SAVE AUDIT
# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

audit_path = os.path.join(
    OUTPUT_DIR,
    f"anti_leakage_audit_{SPLIT_VERSION}.json"
)

manifest_path = os.path.join(
    OUTPUT_DIR,
    f"split_ids_{SPLIT_VERSION}.json"
)

audit["errors"] = errors
audit["warnings"] = warnings
audit["overall_status"] = (
    "PASS" if len(errors) == 0 else "FAIL"
)

# Save the audit even if validation fails.
# Exclusive creation prevents accidental overwriting.
write_json_exclusive(audit_path, audit)

print("\n" + "=" * 65)
print("AUDIT SUMMARY")
print("=" * 65)

print("Errors:", len(errors))
print("Warnings:", len(warnings))

for error in errors:
    print("ERROR:", error)

for warning in warnings:
    print("WARNING:", warning)

print("\nAudit saved to:")
print(audit_path)


# ============================================================
# 13. FREEZE SPLIT IDS ONLY IF AUDIT PASSES
# ============================================================

if errors:
    raise RuntimeError(
        "Split freezing BLOCKED because anti-leakage validation "
        "failed. Review the audit JSON and errors above. "
        "Original datasets were not modified."
    )

manifest = {
    "version": SPLIT_VERSION,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "description": (
        "Frozen image IDs for the knowledge base, validation, "
        "and test splits."
    ),
    "anti_leakage_audit": os.path.basename(audit_path),
    "splits": {
        split_name: {
            "image_count": len(split_ids[split_name]),
            "image_ids": split_ids[split_name],
            "image_id_sha256": sha256_ids(split_ids[split_name]),
        }
        for split_name in split_ids
    },
}

write_json_exclusive(manifest_path, manifest)


# ============================================================
# 14. FINAL RESULT
# ============================================================

print("\n" + "=" * 65)
print("SPLIT FREEZING COMPLETED")
print("=" * 65)

print("Overall status: PASS")
print("Caption text overlap: REVIEW WARNINGS")
print("Image ID overlap: NONE")
print("Filename overlap: NONE")
print("Self-retrieval filter: PASS")

print("\nFrozen split manifest:")
print(manifest_path)

print("\nAnti-leakage audit:")
print(audit_path)

print("\nDataset sizes:")

for split_name in EXPECTED_SIZES:
    print(
        f"{split_name}: "
        f"{len(datasets[split_name]):,} images"
    )

print("\nIMPORTANT:")
print(
    "Use filter_retrieval_candidates(query_image_id, candidate_ids) "
    "in your actual retrieval pipeline to exclude the query image."
)


LOADING DATASETS
kb_10k: 10,000 images loaded
val_100: 100 images loaded
test_1000: 1,000 images loaded

CHECKING DATASET SIZES
kb_10k: 10,000 / 10,000 PASS
val_100: 100 / 100 PASS
test_1000: 1,000 / 1,000 PASS

CHECKING RECORDS

kb_10k:
  invalid_records: 0
  missing_image_ids: 0
  missing_filenames: 0
  duplicate_image_ids: 0
  duplicate_filenames: 0
  invalid_captions: 0
  records_not_having_5_captions: 0
  records_with_duplicate_captions: 18

val_100:
  invalid_records: 0
  missing_image_ids: 0
  missing_filenames: 0
  duplicate_image_ids: 0
  duplicate_filenames: 0
  invalid_captions: 0
  records_not_having_5_captions: 0
  records_with_duplicate_captions: 1

test_1000:
  invalid_records: 0
  missing_image_ids: 0
  missing_filenames: 0
  duplicate_image_ids: 0
  duplicate_filenames: 0
  invalid_captions: 0
  records_not_having_5_captions: 0
  records_with_duplicate_captions: 2

CHECKING IMAGE ID OVERLAP
kb_10k vs val_100: 0 overlapping image IDs PASS
kb_10k vs test_1000: 0 overlapp

In [30]:

# ================================================================
# ENCODER-INDEPENDENT IMAGE PREPROCESSING
# COCO IMAGE RETRIEVAL PROJECT
# ================================================================

!pip -q install pillow tqdm

import os
import json
import hashlib
from datetime import datetime, timezone
from PIL import Image, ImageOps
from tqdm.auto import tqdm

# ================================================================
# 1. CONFIGURATION
# ================================================================

DATASET_PATHS = {
    "kb_10k": "/content/phase_01_data_exploration/splitted_data_set/kb_10k.json",
    "val_100": "/content/phase_01_data_exploration/splitted_data_set/val_100.json",
    "test_100": "/content/phase_01_data_exploration/splitted_data_set/test_100.json",
    "test_1000": "/content/phase_01_data_exploration/splitted_data_set/test_1000.json"
}

IMAGE_DIRS = [
    "/content/train2017",
    "/content/val2017"
]

OUTPUT_DIR = "/content/phase_01_data_exploration/image_preprocessing"

# Keep original dimensions and aspect ratio.
# Do not resize until an encoder is selected.
PRESERVE_ORIGINAL_SIZE = True

# Optional object detection
RUN_OBJECT_DETECTION = True
YOLO_MODEL_NAME = "yolov8n.pt"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ================================================================
# 2. LOAD DATASETS
# ================================================================

print("=" * 65)
print("LOADING COCO DATASETS")
print("=" * 65)

datasets = {}

for split_name, path in DATASET_PATHS.items():

    if not os.path.isfile(path):
        raise FileNotFoundError(path)

    with open(path, "r", encoding="utf-8") as f:
        datasets[split_name] = json.load(f)

    print(split_name, ":", len(datasets[split_name]), "images")


# ================================================================
# 3. FIND IMAGE
# ================================================================

def find_image(file_name):

    if not file_name:
        return None

    for directory in IMAGE_DIRS:

        path = os.path.join(directory, file_name)

        if os.path.isfile(path):
            return path

    return None


# ================================================================
# 4. COMMON IMAGE PREPROCESSING
# ================================================================

def preprocess_image(image_path):

    """
    Encoder-independent preprocessing.

    Operations:
    1. Open image.
    2. Correct EXIF orientation.
    3. Convert to RGB.
    4. Preserve original aspect ratio and dimensions.
    """

    with Image.open(image_path) as img:

        # Correct orientation using EXIF
        img = ImageOps.exif_transpose(img)

        # Convert to RGB
        img = img.convert("RGB")

        # Independent copy
        img = img.copy()

    return img


# ================================================================
# 5. IMAGE METADATA EXTRACTION
# ================================================================

def extract_image_metadata(image, record, image_path):

    width, height = image.size

    return {
        "image_id": str(record["image_id"]),
        "file_name": record["file_name"],
        "image_path": image_path,
        "width": width,
        "height": height,
        "channels": 3,
        "color_mode": "RGB",
        "aspect_ratio": round(width / height, 5),
        "format": "RGB",
        "file_size_bytes": os.path.getsize(image_path)
    }


# ================================================================
# 6. OPTIONAL OBJECT DETECTION
# ================================================================

detector = None

if RUN_OBJECT_DETECTION:

    !pip -q install ultralytics

    from ultralytics import YOLO

    detector = YOLO(YOLO_MODEL_NAME)


def detect_objects(image):

    if detector is None:
        return []

    results = detector.predict(
        source=image,
        verbose=False
    )

    objects = []

    for result in results:

        for box in result.boxes:

            class_id = int(box.cls.item())

            objects.append({
                "class_id": class_id,
                "class_name": detector.names[class_id],
                "confidence": round(float(box.conf.item()), 4),
                "bbox_xyxy": [
                    round(float(v), 2)
                    for v in box.xyxy[0].tolist()
                ]
            })

    return objects


# ================================================================
# 7. PROCESS ALL SPLITS
# ================================================================

print("\n" + "=" * 65)
print("STARTING IMAGE PREPROCESSING")
print("=" * 65)

all_metadata = {}

for split_name, records in datasets.items():

    print(f"\nProcessing {split_name}...")

    split_metadata = []

    missing_images = []
    unreadable_images = []

    for record in tqdm(records):

        image_id = str(record.get("image_id"))
        file_name = record.get("file_name")

        image_path = find_image(file_name)

        if image_path is None:

            missing_images.append({
                "image_id": image_id,
                "file_name": file_name
            })

            continue

        try:

            # RGB + EXIF correction
            image = preprocess_image(image_path)

            # Metadata
            metadata = extract_image_metadata(
                image,
                record,
                image_path
            )

            # Optional object detection
            if RUN_OBJECT_DETECTION:
                metadata["objects"] = detect_objects(image)

            split_metadata.append(metadata)

        except Exception as e:

            unreadable_images.append({
                "image_id": image_id,
                "file_name": file_name,
                "error": str(e)
            })

    all_metadata[split_name] = split_metadata

    print("Successfully processed:", len(split_metadata))
    print("Missing images:", len(missing_images))
    print("Unreadable images:", len(unreadable_images))

    # Save metadata only
    metadata_path = os.path.join(
        OUTPUT_DIR,
        f"{split_name}_metadata.json"
    )

    with open(metadata_path, "w", encoding="utf-8") as f:

        json.dump(
            split_metadata,
            f,
            indent=4,
            ensure_ascii=False
        )

    print("Metadata saved:", metadata_path)

    # Save processing problems only when they exist
    if missing_images or unreadable_images:

        error_path = os.path.join(
            OUTPUT_DIR,
            f"{split_name}_processing_errors.json"
        )

        with open(error_path, "w", encoding="utf-8") as f:

            json.dump(
                {
                    "missing_images": missing_images,
                    "unreadable_images": unreadable_images
                },
                f,
                indent=4,
                ensure_ascii=False
            )


# ================================================================
# 8. PREPROCESSING CONFIGURATION
# ================================================================

preprocessing_config = {
    "version": "v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),

    "encoder": None,

    "encoder_status": "Not selected",

    "operations": [
        "EXIF orientation correction",
        "RGB conversion",
        "Image readability verification",
        "Original dimensions preserved",
        "Original aspect ratio preserved"
    ],

    "resize_applied": False,
    "normalization_applied": False,
    "rescaling_applied": False,

    "object_detection_enabled": RUN_OBJECT_DETECTION,

    "object_detection_model": (
        YOLO_MODEL_NAME if RUN_OBJECT_DETECTION else None
    ),

    "datasets": {
        split_name: {
            "processed_images": len(metadata)
        }
        for split_name, metadata in all_metadata.items()
    }
}

config_path = os.path.join(
    OUTPUT_DIR,
    "preprocessing_config.json"
)

with open(config_path, "w", encoding="utf-8") as f:

    json.dump(
        preprocessing_config,
        f,
        indent=4,
        ensure_ascii=False
    )


# ================================================================
# 9. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 65)
print("PREPROCESSING COMPLETED")
print("=" * 65)

for split_name, metadata in all_metadata.items():

    print(
        split_name,
        ":",
        len(metadata),
        "images processed"
    )

print("\nOutput directory:")
print(OUTPUT_DIR)

print("\nEncoder-specific resizing and normalization:")
print("NOT APPLIED — encoder has not been selected.")

print("\nOriginal images:")
print("UNCHANGED")

print("\nOriginal JSON datasets:")
print("UNCHANGED")

print("\nPreprocessing configuration:")
print(config_path)


LOADING COCO DATASETS
kb_10k : 10000 images
val_100 : 100 images
test_100 : 100 images
test_1000 : 1000 images

STARTING IMAGE PREPROCESSING

Processing kb_10k...


  0%|          | 0/10000 [00:00<?, ?it/s]

Successfully processed: 10000
Missing images: 0
Unreadable images: 0
Metadata saved: /content/phase_01_data_exploration/image_preprocessing/kb_10k_metadata.json

Processing val_100...


  0%|          | 0/100 [00:00<?, ?it/s]

Successfully processed: 100
Missing images: 0
Unreadable images: 0
Metadata saved: /content/phase_01_data_exploration/image_preprocessing/val_100_metadata.json

Processing test_100...


  0%|          | 0/100 [00:00<?, ?it/s]

Successfully processed: 100
Missing images: 0
Unreadable images: 0
Metadata saved: /content/phase_01_data_exploration/image_preprocessing/test_100_metadata.json

Processing test_1000...


  0%|          | 0/1000 [00:00<?, ?it/s]

Successfully processed: 1000
Missing images: 0
Unreadable images: 0
Metadata saved: /content/phase_01_data_exploration/image_preprocessing/test_1000_metadata.json

PREPROCESSING COMPLETED
kb_10k : 10000 images processed
val_100 : 100 images processed
test_100 : 100 images processed
test_1000 : 1000 images processed

Output directory:
/content/phase_01_data_exploration/image_preprocessing

Encoder-specific resizing and normalization:
NOT APPLIED — encoder has not been selected.

Original images:
UNCHANGED

Original JSON datasets:
UNCHANGED

Preprocessing configuration:
/content/phase_01_data_exploration/image_preprocessing/preprocessing_config.json


In [31]:

# ================================================================
# CAPTION PREPROCESSING
# COCO IMAGE RETRIEVAL PROJECT
# ================================================================

import os
import json
import re
import hashlib
from datetime import datetime, timezone
from tqdm.auto import tqdm

# ================================================================
# 1. CONFIGURATION
# ================================================================

DATASET_PATHS = {
    "kb_10k": "/content/phase_01_data_exploration/splitted_data_set/kb_10k.json",
    "val_100": "/content/phase_01_data_exploration/splitted_data_set/val_100.json",
    "test_100": "/content/phase_01_data_exploration/splitted_data_set/test_100.json",
    "test_1000": "/content/phase_01_data_exploration/splitted_data_set/test_1000.json"
}

OUTPUT_DIR = "/content/phase_01_data_exploration/caption_preprocessing"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Caption cleaning options
LOWERCASE = True

NORMALIZE_WHITESPACE = True

# Keep punctuation by default.
# Change to True if your retrieval method requires punctuation removal.
REMOVE_PUNCTUATION = False

# Tokenization pattern:
# Preserves words and internal apostrophes.
TOKEN_PATTERN = r"\b[\w]+(?:['’][\w]+)*\b"

# ================================================================
# 2. LOAD DATASETS
# ================================================================

print("=" * 65)
print("LOADING DATASETS")
print("=" * 65)

datasets = {}

for split_name, path in DATASET_PATHS.items():

    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"Dataset not found: {path}"
        )

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(
            f"{split_name} must contain a JSON list."
        )

    datasets[split_name] = data

    print(
        f"{split_name}: {len(data)} images loaded"
    )


# ================================================================
# 3. CAPTION CLEANING FUNCTION
# ================================================================

def clean_caption(caption):

    """
    Cleans a caption for retrieval/indexing only.

    Original reference captions are never modified.
    """

    if not isinstance(caption, str):
        raise TypeError(
            "Caption must be a string."
        )

    cleaned = caption

    # Step 1: Lowercase
    if LOWERCASE:
        cleaned = cleaned.lower()

    # Step 2: Normalize whitespace
    if NORMALIZE_WHITESPACE:
        cleaned = re.sub(
            r"\s+",
            " ",
            cleaned
        ).strip()

    # Step 3: Optional punctuation removal
    if REMOVE_PUNCTUATION:

        cleaned = re.sub(
            r"[^\w\s]",
            "",
            cleaned
        )

        if NORMALIZE_WHITESPACE:
            cleaned = re.sub(
                r"\s+",
                " ",
                cleaned
            ).strip()

    return cleaned


# ================================================================
# 4. TOKENIZATION FUNCTION
# ================================================================

def tokenize_caption(cleaned_caption):

    """
    Converts cleaned captions into tokens.

    Example:
    'a dog is running in the park'

    becomes:
    ['a', 'dog', 'is', 'running', 'in', 'the', 'park']
    """

    return re.findall(
        TOKEN_PATTERN,
        cleaned_caption,
        flags=re.UNICODE
    )


# ================================================================
# 5. CAPTION HASH FUNCTION
# ================================================================

def caption_hash(caption):

    return hashlib.sha256(
        caption.encode("utf-8")
    ).hexdigest()


# ================================================================
# 6. PROCESS ALL DATASETS
# ================================================================

print("\n" + "=" * 65)
print("STARTING CAPTION PREPROCESSING")
print("=" * 65)

processed_datasets = {}

for split_name, records in datasets.items():

    print(f"\nProcessing {split_name}...")

    processed_records = []

    total_captions = 0
    invalid_captions = 0
    empty_cleaned_captions = 0

    for record in tqdm(records):

        if not isinstance(record, dict):
            raise ValueError(
                f"Invalid record in {split_name}: expected dictionary."
            )

        image_id = record.get("image_id")
        file_name = record.get("file_name")
        raw_captions = record.get("captions")

        if not isinstance(raw_captions, list):
            raise ValueError(
                f"Image {image_id} has invalid captions."
            )

        # Preserve original captions exactly.
        raw_reference_captions = list(raw_captions)

        cleaned_retrieval_captions = []
        retrieval_tokens = []

        for raw_caption in raw_reference_captions:

            total_captions += 1

            if (
                not isinstance(raw_caption, str)
                or not raw_caption.strip()
            ):

                invalid_captions += 1

                raise ValueError(
                    f"Invalid reference caption for image {image_id}."
                )

            # Cleaned caption for retrieval/indexing
            cleaned = clean_caption(raw_caption)

            if not cleaned:
                empty_cleaned_captions += 1

            # Tokenized caption
            tokens = tokenize_caption(cleaned)

            cleaned_retrieval_captions.append(cleaned)
            retrieval_tokens.append(tokens)

        # Create a separate record.
        # Do not overwrite the original dataset.
        processed_record = {
            "image_id": image_id,
            "file_name": file_name,

            # Original evaluation reference captions
            "raw_reference_captions": raw_reference_captions,

            # Cleaned retrieval/indexing captions
            "cleaned_retrieval_captions": cleaned_retrieval_captions,

            # Tokens for retrieval/indexing
            "retrieval_tokens": retrieval_tokens
        }

        processed_records.append(processed_record)

    processed_datasets[split_name] = processed_records

    print("Images processed:", len(processed_records))
    print("Captions processed:", total_captions)
    print("Invalid captions:", invalid_captions)
    print("Empty cleaned captions:", empty_cleaned_captions)


# ================================================================
# 7. VERIFY ORIGINAL CAPTIONS WERE PRESERVED
# ================================================================

print("\n" + "=" * 65)
print("VERIFYING ORIGINAL CAPTIONS")
print("=" * 65)

preservation_results = {}

for split_name in datasets:

    original_records = datasets[split_name]
    processed_records = processed_datasets[split_name]

    passed = True

    for original, processed in zip(
        original_records,
        processed_records
    ):

        original_captions = original["captions"]

        processed_raw_captions = (
            processed["raw_reference_captions"]
        )

        if original_captions != processed_raw_captions:
            passed = False
            break

    preservation_results[split_name] = passed

    print(
        split_name,
        ":",
        "PASS" if passed else "FAIL"
    )

    if not passed:
        raise RuntimeError(
            f"Raw caption preservation failed for {split_name}."
        )


# ================================================================
# 8. GENERATE A PREPROCESSING CONFIGURATION
# ================================================================

preprocessing_config = {

    "version": "v1",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "description": (
        "Caption preprocessing for image retrieval and indexing. "
        "Original reference captions are preserved unchanged."
    ),

    "operations": {

        "lowercase": LOWERCASE,

        "whitespace_normalization": NORMALIZE_WHITESPACE,

        "punctuation_removal": REMOVE_PUNCTUATION,

        "tokenization": True,

        "tokenization_pattern": TOKEN_PATTERN
    },

    "caption_field_definitions": {

        "raw_reference_captions": (
            "Original captions preserved exactly for evaluation."
        ),

        "cleaned_retrieval_captions": (
            "Cleaned captions intended for retrieval and indexing."
        ),

        "retrieval_tokens": (
            "Tokenized cleaned captions intended for retrieval "
            "and indexing."
        )
    },

    "evaluation_policy": (
        "Evaluation must use raw_reference_captions, never "
        "cleaned_retrieval_captions or retrieval_tokens."
    ),

    "raw_caption_preservation": preservation_results,

    "datasets": {
        split_name: {
            "image_count": len(processed_datasets[split_name]),
            "caption_count": sum(
                len(record["raw_reference_captions"])
                for record in processed_datasets[split_name]
            )
        }
        for split_name in processed_datasets
    }
}


# ================================================================
# 9. SAVE PROCESSED DATASETS
# ================================================================

print("\n" + "=" * 65)
print("SAVING PROCESSED CAPTIONS")
print("=" * 65)

for split_name, records in processed_datasets.items():

    output_path = os.path.join(
        OUTPUT_DIR,
        f"{split_name}_captions_v1.json"
    )

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            records,
            f,
            indent=4,
            ensure_ascii=False
        )

    print("Saved:", output_path)


# ================================================================
# 10. SAVE PREPROCESSING CONFIGURATION
# ================================================================

config_path = os.path.join(
    OUTPUT_DIR,
    "caption_preprocessing_config_v1.json"
)

with open(
    config_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        preprocessing_config,
        f,
        indent=4,
        ensure_ascii=False
    )

print("Configuration saved:", config_path)


# ================================================================
# 11. DISPLAY EXAMPLE
# ================================================================

print("\n" + "=" * 65)
print("CAPTION PREPROCESSING EXAMPLE")
print("=" * 65)

example = processed_datasets["kb_10k"][0]

print("\nImage ID:")
print(example["image_id"])

print("\nOriginal reference caption:")
print(example["raw_reference_captions"][0])

print("\nCleaned retrieval caption:")
print(example["cleaned_retrieval_captions"][0])

print("\nRetrieval tokens:")
print(example["retrieval_tokens"][0])


# ================================================================
# 12. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 65)
print("CAPTION PREPROCESSING COMPLETED")
print("=" * 65)

for split_name, records in processed_datasets.items():

    number_of_captions = sum(
        len(record["raw_reference_captions"])
        for record in records
    )

    print(
        f"{split_name}: "
        f"{len(records)} images | "
        f"{number_of_captions} captions"
    )

print("\nRaw reference captions: PRESERVED")
print("Cleaned retrieval captions: CREATED")
print("Retrieval tokens: CREATED")
print("Original JSON files: UNCHANGED")

print("\nOutput directory:")
print(OUTPUT_DIR)


LOADING DATASETS
kb_10k: 10000 images loaded
val_100: 100 images loaded
test_100: 100 images loaded
test_1000: 1000 images loaded

STARTING CAPTION PREPROCESSING

Processing kb_10k...


  0%|          | 0/10000 [00:00<?, ?it/s]

Images processed: 10000
Captions processed: 50000
Invalid captions: 0
Empty cleaned captions: 0

Processing val_100...


  0%|          | 0/100 [00:00<?, ?it/s]

Images processed: 100
Captions processed: 500
Invalid captions: 0
Empty cleaned captions: 0

Processing test_100...


  0%|          | 0/100 [00:00<?, ?it/s]

Images processed: 100
Captions processed: 500
Invalid captions: 0
Empty cleaned captions: 0

Processing test_1000...


  0%|          | 0/1000 [00:00<?, ?it/s]

Images processed: 1000
Captions processed: 5000
Invalid captions: 0
Empty cleaned captions: 0

VERIFYING ORIGINAL CAPTIONS
kb_10k : PASS
val_100 : PASS
test_100 : PASS
test_1000 : PASS

SAVING PROCESSED CAPTIONS
Saved: /content/phase_01_data_exploration/caption_preprocessing/kb_10k_captions_v1.json
Saved: /content/phase_01_data_exploration/caption_preprocessing/val_100_captions_v1.json
Saved: /content/phase_01_data_exploration/caption_preprocessing/test_100_captions_v1.json
Saved: /content/phase_01_data_exploration/caption_preprocessing/test_1000_captions_v1.json
Configuration saved: /content/phase_01_data_exploration/caption_preprocessing/caption_preprocessing_config_v1.json

CAPTION PREPROCESSING EXAMPLE

Image ID:
203564

Original reference caption:
A bicycle replica with a clock as the front wheel.

Cleaned retrieval caption:
a bicycle replica with a clock as the front wheel.

Retrieval tokens:
['a', 'bicycle', 'replica', 'with', 'a', 'clock', 'as', 'the', 'front', 'wheel']

CAPTIO